In [ ]:
import numpy as np
from dataclasses import dataclass

import cvxpy as cp

## CircuitGraph Construction

In [ ]:
@dataclass
class Node:
    op: str
    i: int
    prev: list
    next: list

@dataclass
class Graph:
    nodes: list

In [ ]:
n_qubits = 4
# circuit_data = [
#     ("H", [0]),
#     ("H", [1]),
#     ("H", [2]),
#     ("H", [3]),
#     ("CX", [0,1]),
#     ("CX", [2,3]),
#     ("H", [0]),
#     ("H", [3]),
#     ("CX", [0,3]),
#     ("CX", [1,2]),
#     ("H", [1]),
#     ("H", [2]),
#     ("CX", [1,3]),
#     ("CX", [0,2]),
# ]
circuit_data = [
    ("H", [0]),
    ("X", [1]),
    ("Y", [2]),
    ("Z", [3]),
    ("CX", [0,1]),
    ("CX", [2,3]),
    ("CY", [0,2]),
    ("CZ", [1,3]),
    ("CZ", [0,3]),
    ("CZ", [1,2]),
]

In [ ]:
q_ins_lists = {}
for q in range(n_qubits):
    q_ins_lists[q] = []
    
    for i, ins in enumerate(circuit_data):
        if q in ins[1]:
            q_ins_lists[q].append(i)
print(q_ins_lists)

In [ ]:
graph = Graph(nodes=[])

for i, ins in enumerate(circuit_data):
    node = Node(op=ins[0], i=i, prev=[], next=[])
    graph.nodes.append(node)

for q, q_ins_list in q_ins_lists.items():
    for j in range(len(q_ins_list)):
        if j > 0:
            i = q_ins_list[j]
            im1 = q_ins_list[j-1]
            graph.nodes[i].prev.append(im1)
        if j < len(q_ins_list)-1:
            i = q_ins_list[j]
            ip1 = q_ins_list[j+1]
            graph.nodes[i].next.append(ip1)

print(graph)
print(graph.nodes[1])

## Transport Program Optimization

In [ ]:
# @TODO - do better than this
t_eps = 1

t_list = []
constraints = []

for i in range(len(circuit_data)):
    t_i = cp.Variable(name=f"t_{i}")
    
    t_list.append(t_i)
    constraints.append(t_i >= 0)

for node in graph.nodes:
    i = node.i
    t_i = t_list[i]

    # if len(node.next) > 0:
    #     ip = min(node.next)
    #     t_ip = t_list[ip]
    #     constraints.append(t_ip >= t_i + t_eps)≥
    
    for ip in node.next:
        t_ip = t_list[ip]
        constraints.append(t_ip >= t_i + t_eps)

lp = cp.Problem(
    # cp.Minimize(sum(t_list)), # WRONG (at least, suboptimal)
    cp.Minimize(t_list[-1]), # makes sense but the last object in t_list isn't always the last operation
    constraints
)
lp.solve()

print(lp)
print(f"The optimal value is $\\text{{value}}(f)={lp.value:.2f}$, attained by:")
print(f"\\begin{{itemize}}")
for i in range(len(circuit_data)):
    print(f"\t\\item $t_{i}={t_list[i].value:.2f}$")
print(f"\\end{{itemize}}")